# Batch Scoring — Booking Cancellation Prediction

Scores every booking; writes predictions + per-channel cancellation risk summary.

**Reads:** `gold_ml_features` + saved model  **Writes:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count, isnan, udf,
    sum as spark_sum, round as spark_round
)
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassificationModel

spark = SparkSession.builder.getOrCreate()
model = RandomForestClassificationModel.load('Files/models/cancellation_rf')
df = spark.read.table('gold_ml_features')
print(f'Scoring {df.count():,} feature rows')

In [ ]:
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

numeric_features = [
    'nights', 'room_rate', 'lead_time_days', 'is_refundable',
    'total_stays', 'total_spend', 'star_rating', 'room_count',
]
cat_cols = ['room_type', 'channel', 'meal_plan', 'loyalty_tier', 'region', 'age_group', 'property_type']
indexed_df = df
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)

all_features = numeric_features + [f'{c}_idx' for c in cat_cols]
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(indexed_df)

In [ ]:
scored = model.transform(model_df)
extract_prob = udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.0, DoubleType())

predictions = (
    scored
    .withColumn('cancel_probability', spark_round(extract_prob(col('probability')), 4))
    .withColumn('predicted_cancel', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('cancel_probability') > 0.8, 'critical')
        .when(col('cancel_probability') > 0.6, 'high')
        .when(col('cancel_probability') > 0.4, 'medium')
        .otherwise('low'))
    .withColumn('scored_at', current_timestamp())
    .select(
        'booking_id', 'guest_id', 'property_id', 'channel', 'room_type', 'loyalty_tier',
        'room_rate', 'lead_time_days', 'nights',
        'had_cancel', 'predicted_cancel', 'cancel_probability', 'risk_level',
        'scored_at')
)
predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count():,} rows')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Per-channel cancellation risk summary
summary = (
    predictions
    .groupBy('channel')
    .agg(
        count('*').alias('total_bookings'),
        spark_sum('predicted_cancel').alias('predicted_cancel_count'),
        spark_sum('had_cancel').alias('actual_cancel_count'),
        spark_round(avg('cancel_probability'), 4).alias('avg_cancel_probability'),
        spark_round(avg('room_rate'), 2).alias('avg_room_rate'),
        spark_round(avg('lead_time_days'), 1).alias('avg_lead_time_days'),
    )
    .withColumn('cancel_rate', spark_round(col('predicted_cancel_count') / col('total_bookings') * 100, 1))
    .withColumn('overall_risk',
        when(col('avg_cancel_probability') > 0.6, 'high')
        .when(col('avg_cancel_probability') > 0.3, 'medium')
        .otherwise('low'))
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_cancel_probability').desc())
)
summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'Channel cancellation summary: {summary.count()} rows')
summary.show(15, truncate=False)

In [ ]:
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('All Gold ML tables optimized')